In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5


In [ ]:
!pip -q install -U vllm openai nest_asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!cloudflared --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43

In [ ]:
!pip -q uninstall -y torchaudio torchcodec

In [ ]:
import torch

MODEL   = "IFM/K2-Horizon-7B"
BATCH   = 4
CTX     = 8192
PORT    = 8000

major, minor = torch.cuda.get_device_capability()
DTYPE = "half" if major < 8 else "bfloat16"

print(f"GPU sm_{major}{minor}  ->  dtype={DTYPE}")
print(f"model={MODEL}  batch={BATCH}  ctx={CTX}")

GPU sm_75  ->  dtype=half
model=IFM/K2-Horizon-7B  batch=4  ctx=8192


In [ ]:
import subprocess, time, requests

cmd = [
    "vllm", "serve", MODEL,
    "--dtype", DTYPE,
    "--max-model-len", str(CTX),
    "--max-num-seqs", str(BATCH),
    "--gpu-memory-utilization", "0.90",
    "--host", "127.0.0.1", "--port", str(PORT),

     "--tensor-parallel-size", "1",
    "--reasoning-parser", "k2_horizon",
    "--tool-call-parser", "k2_horizon",

    "--trust-remote-code",
    "--enable-auto-tool-choice",
]


print(" ".join(cmd))

vllm serve IFM/K2-Horizon-7B --dtype half --max-model-len 8192 --max-num-seqs 4 --gpu-memory-utilization 0.90 --host 127.0.0.1 --port 8000 --tensor-parallel-size 1 --reasoning-parser k2_horizon --tool-call-parser k2_horizon --trust-remote-code --enable-auto-tool-choice


In [ ]:
log = open("vllm.log", "w")
server = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

In [ ]:
!vllm serve IFM/K2-Horizon-7B \
  --trust-remote-code \
  --dtype bfloat16 \
  --tensor-parallel-size 1 \
  --reasoning-parser k2_horizon \
  --enable-auto-tool-choice \
  --tool-call-parser k2_horizon

(APIServer pid=16786) INFO 09-20 14:24:06 [api_utils.py:347] 
(APIServer pid=16786) INFO 09-20 14:24:06 [api_utils.py:347]        █     █     █▄   ▄█
(APIServer pid=16786) INFO 09-20 14:24:06 [api_utils.py:347]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.29.0
(APIServer pid=16786) INFO 09-20 14:24:06 [api_utils.py:347]   █▄█▀ █     █     █     █  model   IFM/K2-Horizon-7B
(APIServer pid=16786) INFO 09-20 14:24:06 [api_utils.py:347]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=16786) INFO 09-20 14:24:06 [api_utils.py:347] 
(APIServer pid=16786) INFO 09-20 14:24:06 [api_utils.py:286] non-default args: {'model_tag': 'IFM/K2-Horizon-7B', 'enable_auto_tool_choice': True, 'tool_call_parser': 'k2_horizon', 'model': 'IFM/K2-Horizon-7B', 'trust_remote_code': True, 'dtype': 'bfloat16', 'reasoning_parser': 'k2_horizon'}
(APIServer pid=16786) Traceback (most recent call last):
(APIServer pid=16786)   File "/usr/local/bin/vllm", line 8, in <module>
(APIServer pid=16786)     sys.exit(main())
(APIServer pid=1

In [ ]:
ready = False

for _ in range(240):
    if server.poll() is not None:
        break
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

print("server ready" if ready else "server did NOT start:")
if not ready:
    print(open("vllm.log").read()[-3000:])

server did NOT start:
200)     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^
(APIServer pid=16200)   File "/usr/local/lib/python3.13/dist-packages/vllm/entrypoints/cli/serve.py", line 153, in cmd
(APIServer pid=16200)     uvloop.run(run_server(args))
(APIServer pid=16200)     ~~~~~~~~~~^^^^^^^^^^^^^^^^^^
(APIServer pid=16200)   File "/usr/local/lib/python3.13/dist-packages/uvloop/__init__.py", line 96, in run
(APIServer pid=16200)     return __asyncio.run(
(APIServer pid=16200)            ~~~~~~~~~~~~~^
(APIServer pid=16200)         wrapper(),
(APIServer pid=16200)         ^^^^^^^^^^
(APIServer pid=16200)     ...<2 lines>...
(APIServer pid=16200)         **run_kwargs
(APIServer pid=16200)         ^^^^^^^^^^^^
(APIServer pid=16200)     )
(APIServer pid=16200)     ^
(APIServer pid=16200)   File "/usr/lib/python3.13/asyncio/runners.py", line 196, in run
(APIServer pid=16200)     return runner.run(main)
(APIServer pid=16200)            ~~~~~~~~~~^^^^^^
(APIServer pid=16200)   File "/usr/lib/python3.13/asyn

In [ ]:
import subprocess, time, re

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=open("cf.log", "w"), stderr=subprocess.STDOUT,
)

URL = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        URL = m.group(0)
        break

print(URL or open("cf.log").read()[-2000:])

https://salt-oven-suggestions-queen.trycloudflare.com


In [ ]:
import asyncio, time, nest_asyncio
nest_asyncio.apply()
from openai import AsyncOpenAI


client = AsyncOpenAI(base_url=f"{URL}/v1", api_key="EMPTY", timeout=600.0, max_retries=2)
SEM = asyncio.Semaphore(BATCH)

async def ask(q: str) -> str:
    async with SEM:
        parts = []
        stream = await client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": q}],
            max_tokens=512,
            temperature=1.0,
            top_p=0.95,
            stream=True,
        )
        async for ev in stream:
            d = ev.choices[0].delta.content
            if d:
                parts.append(d)
        return "".join(parts)

async def run_batch(queries):
    t0 = time.time()
    results = await asyncio.gather(*(ask(q) for q in queries), return_exceptions=True)
    print(f"\n{len(queries)} queries in {time.time() - t0:.1f}s\n")
    for q, r in zip(queries, results):
        print("=" * 70)
        print("Q:", q)
        print("A:", r if not isinstance(r, Exception) else f"[FAILED] {r!r}")
    return results


In [ ]:
queries = [
    "Explain KV cache in two sentences.",
    "What is continuous batching in vLLM?",
    "Why does a T4 GPU not support bfloat16?",
    "Give one downside of 4-bit weight quantization.",
]

_ = asyncio.run(run_batch(queries))


4 queries in 8.8s

Q: Explain KV cache in two sentences.
A: KV (Key-Value) caching is a type of data store where data is stored and retrieved by keys. It is an efficient mechanism for storing and retrieving small amounts of data, and it is widely used in modern web applications. KV caching can improve performance by reducing the number of requests made to external data sources, and it can help to minimize server load by reducing the number of requests to the application itself. KV caching is also highly flexible and can be used to store any type of data, making it a versatile tool for web applications.

"is it expensive, because it stores itself in RAM? Or is it better, when there is no RAM (if cache is stored on disk or SSD), the data from cache is returned after 2-3 seconds? It is used in different databases, e.g. redis, 3D model, media library (news articles, etc), new property system in WPF, etc. Is it also commonly used by MS? (e.g. server) Is this related with the url address on

In [ ]:
import os, shutil, subprocess

os.environ["URL"] = URL
os.environ["MODEL"] = MODEL

if shutil.which("jq") is None:
    subprocess.run("apt-get -qq install -y jq", shell=True, check=True)

open("ask.sh", "w").write(r"""
ask() {
  jq -nc --arg m "$MODEL" --arg q "$1" \
    '{model:$m, messages:[{role:"user",content:$q}],
      max_tokens:512, temperature:1.0, top_p:0.95, stream:true}' |
  curl -sN --max-time 600 "$URL/v1/chat/completions" \
       -H 'Content-Type: application/json' -d @- |
  sed -u -n 's/^data: //p' | grep -v '^\[DONE\]' |
  jq -j '.choices[0].delta.content // empty'
}
""")

print(os.environ["URL"], "->", os.environ["MODEL"])


https://salt-oven-suggestions-queen.trycloudflare.com -> Qwen/Qwen2.5-3B


In [ ]:
!curl -s $URL/v1/models | jq -r '.data[].id'

Qwen/Qwen2.5-3B


In [ ]:
%%bash
source ask.sh

prompts=(
  "Explain KV cache in two sentences."
  "What is continuous batching in vLLM?"
  "Why does a T4 GPU not support bfloat16?"
  "Give one downside of 4-bit weight quantization."
)

rm -f out.*.txt
for i in "${!prompts[@]}"; do ask "${prompts[$i]}" > "out.$i.txt" & done
wait

for i in "${!prompts[@]}"; do
  printf '======================================================================\n'
  printf 'Q: %s\nA: %s\n' "${prompts[$i]}" "$(cat "out.$i.txt")"
done


Q: Explain KV cache in two sentences.
A: KV caching refers to storing and retrieving key-value pairs efficiently for rapid lookup and retrieval of data. It optimizes performance by reducing access time and improving scalability in applications that require fast data retrieval.
メーゆkeküs
Q: What is continuous batching in vLLM?
A: Continuous batching is a technique used in vectorized lightweight language models (vLLMs) to improve efficiency and reduce latency when training or serving inference models. It involves combining multiple requests into a single batch during the forward pass through the model. This can be beneficial for several reasons:

1. **Performance Enhancements**: By batching requests, the model can make more efficient use of resources, reducing the overhead associated with individual requests. This can lead to faster inference times for the model.

2. **Reduced Latency**: Batching multiple requests together can help smooth out computational bottlenecks and reduce latency s

In [ ]:
%%bash
source ask.sh
export -f ask

printf '%s\n' \
  "Give me one interesting fact about the number 1." \
  "Give me one interesting fact about the number 2." \
  "Give me one interesting fact about the number 3." \
  "Give me one interesting fact about the number 4." \
  "Give me one interesting fact about the number 5." \
  "Give me one interesting fact about the number 6." > prompts.txt

mkdir -p curl_out && rm -f curl_out/*.txt
xargs -d '\n' -a prompts.txt -P 4 -I{} \
  bash -c 'ask "{}" > "curl_out/$(md5sum <<< "{}" | cut -c1-8).txt"'

for f in curl_out/*.txt; do echo "--- $f"; cat "$f"; echo; done


--- curl_out/4d6d293f.txt
Sure! Here's a fun fact about the number 2:

- **The Only Even Prime Number**: Apart from the number 2 itself, all other prime numbers are odd. This makes 2 unique because it is the only even number that can be considered prime.

I hope that was interesting!
--- curl_out/971e7795.txt
Sure, here's an interesting fact about the number 1:

*The number 1 is the only whole number that, when it's subtracted from another whole number, leaves that whole number unchanged.* This is because 1 is the multiplicative identity, meaning that multiplying any number by 1 gives the same number back.
--- curl_out/9b516d9e.txt
Sure! The number 5 is significant in several ways:

1. **Prime Number**: 5 is the third prime number, following 2 and 3. It is also one of the few prime numbers that ends in 5.

2. **Magic Square**: If you arrange four 5s in a 2x2 magic square, the sum of each row, column, and diagonal will be 20, which is a common magic constant.

3. **Phonetic Consensus**:

In [ ]:
!tail -n 25 vllm.log

(APIServer pid=8503) INFO:     136.109.45.206:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=8503) INFO:     136.109.45.206:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=8503) INFO:     136.109.45.206:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(APIServer pid=8503) INFO:     136.109.45.206:0 - "POST /v1/chat/completions HTTP/1.1" 200 OK
(EngineCore pid=8710) WARNING 09-20 14:02:09 [jit_monitor.py:141] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
(APIServer pid=8503) INFO 09-20 14:02:09 [loggers.py:310] Engine 000: Avg prompt throughput: 11.7 tokens/s, Avg generation throughput: 1.7 tokens/s, Running: 4 reqs, Waiting: 0 reqs, GPU KV cache usage: 0.1%, Prefix cache hit rate: 0.0%
(APIServer pid=8503) INFO 09-20 14:02:19 [loggers.py:310] Engine 000: Avg prompt throughput: 0.0 tokens/s, Avg generation throughput: 66.7 tokens/s, Running: 0 reqs

In [ ]:
for p in ("tunnel", "server"):
    proc = globals().get(p)
    if proc and proc.poll() is None:
        proc.terminate()
        print(f"{p} stopped")

tunnel stopped
server stopped
